# Advanced Python Dictionary Problems — Tutorial-Style Solutions

This notebook develops advanced dictionary skills through a new set of realistic problems.

The emphasis is not just on obtaining the final answer. Each problem is broken into logical steps:

1. understand the data,
2. choose an appropriate dictionary representation,
3. build the solution incrementally,
4. test intermediate results,
5. inspect edge cases,
6. finish with a reusable implementation.

The main dictionary operations used throughout are:

- direct lookup with `d[key]`,
- safe lookup with `d.get(key, default)`,
- membership tests with `in` and `not in`,
- insertion with assignment and `setdefault`,
- removal with `del`, `pop`, and `popitem`,
- complete reset with `clear`,
- nested dictionaries,
- dictionaries containing lists, sets, tuples, and other dictionaries.

All examples use only the Python standard library.

## Best-practice checklist

Before solving a dictionary problem, ask:

- What should each key represent?
- What should each value represent?
- Is a missing key normal or exceptional?
- Should duplicate values be preserved, counted, or removed?
- Does insertion order matter?
- Will the function mutate the input dictionary?
- Do I need a shallow copy or a nested/deep copy?
- What invariants must remain true after an update?
- What is the expected time complexity?

A clear dictionary design often makes the rest of the algorithm straightforward.

In [1]:
from __future__ import annotations

from collections.abc import Iterable, Mapping
from copy import deepcopy
from pprint import pprint
import math
import re

## Small testing helper

We will use ordinary `assert` statements throughout the notebook.

The helper below makes some comparisons easier to read and gives a useful message when a test fails.

In [2]:
def assert_equal(actual, expected, message=""):
    assert actual == expected, (
        f"{message}\nExpected: {expected!r}\nActual:   {actual!r}"
    )

# Problem 1 — Build a normalized token-frequency table

We are given several text fragments. We want to count normalized words.

The rules are:

- comparisons are case-insensitive,
- punctuation is ignored,
- apostrophes inside words are preserved,
- empty tokens are ignored,
- the original input strings must not be modified.

This problem develops:

- `get`,
- membership,
- incremental counting,
- dictionary comprehensions,
- deterministic sorting.

## Step 1 — Inspect the raw data

A useful first step is to look at the exact text rather than immediately writing the counting loop.

In [3]:
documents = [
    "Python's dictionaries are fast, flexible, and readable.",
    "Readable Python often uses dictionaries.",
    "FAST code is useful, but clear code is easier to maintain.",
]

for index, document in enumerate(documents, start=1):
    print(f"Document {index}: {document}")

Document 1: Python's dictionaries are fast, flexible, and readable.
Document 2: Readable Python often uses dictionaries.
Document 3: FAST code is useful, but clear code is easier to maintain.


## Step 2 — Write one normalization function

Keeping normalization in a separate function gives us one place to test and change the tokenization rules.

The regular expression below accepts alphabetic words and optional apostrophe-separated parts.

In [4]:
TOKEN_PATTERN = re.compile(r"[A-Za-z]+(?:'[A-Za-z]+)*")

def normalize_tokens(text: str) -> list[str]:
    return [match.group(0).lower() for match in TOKEN_PATTERN.finditer(text)]

Before counting every document, test the tokenizer on a small example.

In [5]:
sample_tokens = normalize_tokens("Python's fast—really fast!")
sample_tokens

["python's", 'fast', 'really', 'fast']

In [6]:
assert_equal(sample_tokens, ["python's", "fast", "really", "fast"])

## Step 3 — Count with `get`

The expression

```python
counts[token] = counts.get(token, 0) + 1
```

means:

1. retrieve the existing count,
2. use `0` when the token is not present,
3. add one,
4. store the new count.

In [7]:
counts = {}

for document in documents:
    for token in normalize_tokens(document):
        counts[token] = counts.get(token, 0) + 1

pprint(counts)

{'and': 1,
 'are': 1,
 'but': 1,
 'clear': 1,
 'code': 2,
 'dictionaries': 2,
 'easier': 1,
 'fast': 2,
 'flexible': 1,
 'is': 2,
 'maintain': 1,
 'often': 1,
 'python': 1,
 "python's": 1,
 'readable': 2,
 'to': 1,
 'useful': 1,
 'uses': 1}


## Step 4 — Separate repeated and unique tokens

A dictionary comprehension is convenient when the dictionary has already been built and we want filtered views.

In [8]:
repeated = {token: count for token, count in counts.items() if count > 1}
singletons = {token: count for token, count in counts.items() if count == 1}

print("Repeated:")
pprint(repeated)

print("\nSingletons:")
pprint(singletons)

Repeated:
{'code': 2, 'dictionaries': 2, 'fast': 2, 'is': 2, 'readable': 2}

Singletons:
{'and': 1,
 'are': 1,
 'but': 1,
 'clear': 1,
 'easier': 1,
 'flexible': 1,
 'maintain': 1,
 'often': 1,
 'python': 1,
 "python's": 1,
 'to': 1,
 'useful': 1,
 'uses': 1}


## Step 5 — Produce a deterministic ranking

Dictionaries preserve insertion order, but insertion order is not the same as frequency order.

For a stable report, sort by:

1. descending count,
2. ascending token for ties.

In [9]:
ranking = sorted(counts.items(), key=lambda item: (-item[1], item[0]))
ranking[:10]

[('code', 2),
 ('dictionaries', 2),
 ('fast', 2),
 ('is', 2),
 ('readable', 2),
 ('and', 1),
 ('are', 1),
 ('but', 1),
 ('clear', 1),
 ('easier', 1)]

## Complete reusable solution

In [10]:
def token_frequencies(texts: Iterable[str]) -> dict[str, int]:
    frequencies: dict[str, int] = {}

    for text in texts:
        for token in normalize_tokens(text):
            frequencies[token] = frequencies.get(token, 0) + 1

    return frequencies


def rank_frequencies(frequencies: Mapping[str, int]) -> list[tuple[str, int]]:
    return sorted(frequencies.items(), key=lambda item: (-item[1], item[0]))

In [11]:
result = token_frequencies(documents)

assert_equal(result["dictionaries"], 2)
assert_equal(result["fast"], 2)
assert_equal(result["code"], 2)
assert_equal(result["python's"], 1)

rank_frequencies(result)[:6]

[('code', 2),
 ('dictionaries', 2),
 ('fast', 2),
 ('is', 2),
 ('readable', 2),
 ('and', 1)]

### Best-practice note

Use `get` when a missing key naturally means a default value.

Do not use `get` when a missing key signals corrupted data or a programming error. In that case, direct indexing may be more appropriate because the resulting `KeyError` exposes the problem.

# Problem 2 — Resolve layered configuration safely

Applications often combine configuration from several sources:

1. built-in defaults,
2. a project file,
3. environment-specific overrides,
4. runtime overrides.

Later layers should override earlier layers, but only known settings are allowed.

This problem develops:

- membership checks,
- dictionary copying,
- update logic,
- distinguishing a missing key from a key whose value is `None`,
- validation before mutation.

In [12]:
defaults = {
    "host": "localhost",
    "port": 8000,
    "debug": False,
    "timeout": 30,
}

project_config = {
    "port": 9000,
    "timeout": 45,
}

environment_config = {
    "debug": True,
}

runtime_config = {
    "timeout": None,
}

## Step 1 — Understand the `None` issue

A value of `None` may be an intentional override. Therefore, this is unsafe:

```python
value = layer.get(key)
if value is not None:
    ...
```

That code cannot distinguish:

- key missing,
- key present with value `None`.

Membership tests solve this problem.

In [13]:
print("timeout" in runtime_config)
print(runtime_config["timeout"])
print("missing" in runtime_config)

True
None
False


## Step 2 — Validate every layer before applying it

Validation should happen before the result is changed. This avoids returning a partially merged configuration after an invalid key is encountered.

In [14]:
allowed_keys = set(defaults)

layers = [project_config, environment_config, runtime_config]

for layer_number, layer in enumerate(layers, start=1):
    unknown = set(layer) - allowed_keys
    print(f"Layer {layer_number} unknown keys:", unknown)

Layer 1 unknown keys: set()
Layer 2 unknown keys: set()
Layer 3 unknown keys: set()


## Step 3 — Merge into a fresh dictionary

We should not mutate `defaults`, because defaults may be reused for another configuration.

In [15]:
resolved = defaults.copy()

for layer in layers:
    for key, value in layer.items():
        resolved[key] = value

resolved

{'host': 'localhost', 'port': 9000, 'debug': True, 'timeout': None}

In [16]:
assert_equal(defaults["port"], 8000, "The defaults dictionary was mutated.")
assert_equal(resolved["port"], 9000)
assert_equal(resolved["debug"], True)
assert resolved["timeout"] is None

## Complete reusable solution

In [17]:
def resolve_configuration(
    defaults: Mapping[str, object],
    *layers: Mapping[str, object],
) -> dict[str, object]:
    allowed = set(defaults)

    for layer_index, layer in enumerate(layers, start=1):
        unknown = set(layer) - allowed
        if unknown:
            names = ", ".join(sorted(unknown))
            raise KeyError(f"Unknown configuration key(s) in layer {layer_index}: {names}")

    result = dict(defaults)

    for layer in layers:
        result.update(layer)

    return result

In [18]:
resolved = resolve_configuration(
    defaults,
    project_config,
    environment_config,
    runtime_config,
)

pprint(resolved)

{'debug': True, 'host': 'localhost', 'port': 9000, 'timeout': None}


## Step 4 — Test failure without partial mutation

In [19]:
bad_layer = {"port": 7000, "colour_scheme": "dark"}

try:
    resolve_configuration(defaults, project_config, bad_layer)
except KeyError as exc:
    print(exc)

print("Defaults are still unchanged:")
pprint(defaults)

'Unknown configuration key(s) in layer 2: colour_scheme'
Defaults are still unchanged:
{'debug': False, 'host': 'localhost', 'port': 8000, 'timeout': 30}


### Variation — Ignore unknown keys explicitly

Silently ignoring unknown keys is sometimes useful, but it should be an explicit policy rather than an accidental behavior.

In [20]:
def resolve_known_configuration(
    defaults: Mapping[str, object],
    *layers: Mapping[str, object],
) -> dict[str, object]:
    result = dict(defaults)

    for layer in layers:
        for key, value in layer.items():
            if key in result:
                result[key] = value

    return result


resolve_known_configuration(defaults, bad_layer)

{'host': 'localhost', 'port': 7000, 'debug': False, 'timeout': 30}

# Problem 3 — Build a nested log summary with `setdefault`

Each log record has:

- a service,
- a severity,
- a message.

We want a nested dictionary with this shape:

```python
{
    service: {
        severity: [messages...]
    }
}
```

This problem develops nested dictionaries and explains exactly what each `setdefault` call returns.

In [21]:
logs = [
    {"service": "api", "severity": "INFO", "message": "Request accepted"},
    {"service": "worker", "severity": "ERROR", "message": "Job failed"},
    {"service": "api", "severity": "WARNING", "message": "Slow response"},
    {"service": "api", "severity": "ERROR", "message": "Database unavailable"},
    {"service": "worker", "severity": "INFO", "message": "Retry scheduled"},
    {"service": "api", "severity": "ERROR", "message": "Request timed out"},
]

## Step 1 — Group by service only

Start with one level. This makes it easier to verify the representation.

In [22]:
by_service = {}

for record in logs:
    service = record["service"]
    by_service.setdefault(service, []).append(record)

pprint(by_service)

{'api': [{'message': 'Request accepted', 'service': 'api', 'severity': 'INFO'},
         {'message': 'Slow response', 'service': 'api', 'severity': 'WARNING'},
         {'message': 'Database unavailable',
          'service': 'api',
          'severity': 'ERROR'},
         {'message': 'Request timed out',
          'service': 'api',
          'severity': 'ERROR'}],
 'worker': [{'message': 'Job failed', 'service': 'worker', 'severity': 'ERROR'},
            {'message': 'Retry scheduled',
             'service': 'worker',
             'severity': 'INFO'}]}


`setdefault(service, [])` performs two possible actions:

- if `service` exists, return the existing list;
- otherwise, insert a new empty list and return that list.

The returned list is then mutated with `.append(record)`.

## Step 2 — Introduce the severity level

For every record, we need:

1. the dictionary for the service,
2. the list for the severity,
3. append the message.

In [23]:
summary = {}

for record in logs:
    service = record["service"]
    severity = record["severity"]
    message = record["message"]

    service_bucket = summary.setdefault(service, {})
    message_bucket = service_bucket.setdefault(severity, [])
    message_bucket.append(message)

pprint(summary)

{'api': {'ERROR': ['Database unavailable', 'Request timed out'],
         'INFO': ['Request accepted'],
         'WARNING': ['Slow response']},
 'worker': {'ERROR': ['Job failed'], 'INFO': ['Retry scheduled']}}


## Step 3 — Write the compact equivalent

The compact form is useful after the logic is understood.

In [24]:
compact_summary = {}

for record in logs:
    compact_summary        .setdefault(record["service"], {})        .setdefault(record["severity"], [])        .append(record["message"])

assert_equal(compact_summary, summary)

## Step 4 — Add counts without losing messages

We now want each severity bucket to contain both:

- `count`,
- `messages`.

In [25]:
detailed_summary = {}

for record in logs:
    bucket = (
        detailed_summary
        .setdefault(record["service"], {})
        .setdefault(
            record["severity"],
            {"count": 0, "messages": []},
        )
    )

    bucket["count"] += 1
    bucket["messages"].append(record["message"])

pprint(detailed_summary)

{'api': {'ERROR': {'count': 2,
                   'messages': ['Database unavailable', 'Request timed out']},
         'INFO': {'count': 1, 'messages': ['Request accepted']},
         'WARNING': {'count': 1, 'messages': ['Slow response']}},
 'worker': {'ERROR': {'count': 1, 'messages': ['Job failed']},
            'INFO': {'count': 1, 'messages': ['Retry scheduled']}}}


## Complete reusable solution

In [26]:
def summarize_logs(
    records: Iterable[Mapping[str, str]],
) -> dict[str, dict[str, dict[str, object]]]:
    result: dict[str, dict[str, dict[str, object]]] = {}

    for record in records:
        service = record["service"]
        severity = record["severity"]
        message = record["message"]

        severity_bucket = (
            result
            .setdefault(service, {})
            .setdefault(severity, {"count": 0, "messages": []})
        )

        severity_bucket["count"] += 1
        severity_bucket["messages"].append(message)

    return result

In [27]:
log_summary = summarize_logs(logs)

assert_equal(log_summary["api"]["ERROR"]["count"], 2)
assert_equal(
    log_summary["worker"]["INFO"]["messages"],
    ["Retry scheduled"],
)

pprint(log_summary)

{'api': {'ERROR': {'count': 2,
                   'messages': ['Database unavailable', 'Request timed out']},
         'INFO': {'count': 1, 'messages': ['Request accepted']},
         'WARNING': {'count': 1, 'messages': ['Slow response']}},
 'worker': {'ERROR': {'count': 1, 'messages': ['Job failed']},
            'INFO': {'count': 1, 'messages': ['Retry scheduled']}}}


### Common pitfall — Shared mutable defaults

This is dangerous:

```python
shared = []
d = dict.fromkeys(["a", "b"], shared)
```

Both keys reference the same list.

In [28]:
shared = []
bad = dict.fromkeys(["a", "b"], shared)
bad["a"].append("unexpected")

pprint(bad)
assert bad["a"] is bad["b"]

{'a': ['unexpected'], 'b': ['unexpected']}


Create a separate mutable value for each key instead.

In [29]:
good = {key: [] for key in ["a", "b"]}
good["a"].append("only in a")

pprint(good)
assert good["a"] is not good["b"]

{'a': ['only in a'], 'b': []}


# Problem 4 — Add and multiply sparse vectors

A sparse vector stores only non-zero coordinates:

```python
{index: value}
```

For example:

```python
{0: 3, 4: -2}
```

represents a vector whose coordinate `0` is `3`, coordinate `4` is `-2`, and all omitted coordinates are zero.

This problem develops:

- `get` for implicit zero values,
- iterating over keys efficiently,
- removing zero results with `pop`,
- representation invariants.

In [30]:
u = {0: 3.0, 2: 4.0, 7: -1.0}
v = {1: 5.0, 2: -4.0, 7: 2.0}

## Step 1 — Read an omitted coordinate

A missing coordinate means zero, so `get(index, 0.0)` is the natural operation.

In [31]:
for index in range(4):
    print(index, u.get(index, 0.0))

0 3.0
1 0.0
2 4.0
3 0.0


## Step 2 — Add one vector into another copy

We begin with a copy of `u`, then add every non-zero coordinate from `v`.

In [32]:
sum_vector = dict(u)

for index, value in v.items():
    sum_vector[index] = sum_vector.get(index, 0.0) + value

sum_vector

{0: 3.0, 2: 0.0, 7: 1.0, 1: 5.0}

Coordinate `2` became zero. A sparse representation should not store explicit zeros, so we clean it up.

In [33]:
for index in list(sum_vector):
    if math.isclose(sum_vector[index], 0.0, abs_tol=1e-12):
        sum_vector.pop(index)

sum_vector

{0: 3.0, 7: 1.0, 1: 5.0}

Why use `list(sum_vector)`?

Because removing keys while directly iterating over a dictionary changes its size and raises a runtime error. Iterating over a list of the keys creates a stable snapshot.

## Step 3 — Implement sparse addition cleanly

In [34]:
def add_sparse_vectors(
    left: Mapping[int, float],
    right: Mapping[int, float],
    *,
    zero_tolerance: float = 1e-12,
) -> dict[int, float]:
    result = dict(left)

    for index, value in right.items():
        new_value = result.get(index, 0.0) + value

        if math.isclose(new_value, 0.0, abs_tol=zero_tolerance):
            result.pop(index, None)
        else:
            result[index] = new_value

    return result

In [35]:
added = add_sparse_vectors(u, v)
assert_equal(added, {0: 3.0, 1: 5.0, 7: 1.0})
added

{0: 3.0, 7: 1.0, 1: 5.0}

## Step 4 — Compute a sparse dot product

For the dot product, only coordinates that occur in both vectors contribute.

Iterate over the smaller dictionary to reduce the number of membership tests.

In [36]:
def sparse_dot(
    left: Mapping[int, float],
    right: Mapping[int, float],
) -> float:
    if len(left) > len(right):
        left, right = right, left

    total = 0.0

    for index, value in left.items():
        if index in right:
            total += value * right[index]

    return total

In [37]:
dot = sparse_dot(u, v)
print(dot)

# 4 * -4 + (-1) * 2 = -18
assert_equal(dot, -18.0)

-18.0


### Complexity discussion

Let `m` and `n` be the numbers of stored non-zero coordinates.

- addition is approximately `O(m + n)`,
- dot product is `O(min(m, n))` average-case dictionary lookups,
- storage is proportional to the number of non-zero coordinates, not the full vector length.

# Problem 5 — Implement an inventory reservation transaction

An order requests quantities from inventory.

The reservation must be atomic:

- either every item is reserved,
- or the inventory remains unchanged.

This problem develops:

- validation before mutation,
- copying,
- safe lookup,
- removing zero-quantity items with `pop`,
- returning structured results.

In [38]:
inventory = {
    "keyboard": 5,
    "mouse": 8,
    "monitor": 3,
    "webcam": 2,
}

order = {
    "keyboard": 2,
    "mouse": 1,
    "webcam": 2,
}

## Step 1 — Check the order without changing inventory

A reservation can fail because:

- a product does not exist,
- a requested quantity is not positive,
- insufficient stock is available.

In [39]:
problems = {}

for product, requested in order.items():
    if requested <= 0:
        problems[product] = "quantity must be positive"
    elif product not in inventory:
        problems[product] = "unknown product"
    elif inventory[product] < requested:
        problems[product] = (
            f"requested {requested}, available {inventory[product]}"
        )

problems

{}

## Step 2 — Apply changes to a working copy

A copy gives us transaction-like behavior. The original is untouched until all checks succeed.

In [40]:
working_inventory = inventory.copy()

for product, requested in order.items():
    remaining = working_inventory[product] - requested

    if remaining == 0:
        working_inventory.pop(product)
    else:
        working_inventory[product] = remaining

working_inventory

{'keyboard': 3, 'mouse': 7, 'monitor': 3}

In [41]:
assert_equal(
    inventory,
    {"keyboard": 5, "mouse": 8, "monitor": 3, "webcam": 2},
    "The original inventory should remain unchanged.",
)

## Step 3 — Complete reusable solution

In [42]:
def reserve_inventory(
    inventory: Mapping[str, int],
    order: Mapping[str, int],
) -> tuple[dict[str, int], dict[str, str]]:
    errors: dict[str, str] = {}

    for product, requested in order.items():
        if requested <= 0:
            errors[product] = "quantity must be positive"
        elif product not in inventory:
            errors[product] = "unknown product"
        elif inventory[product] < requested:
            errors[product] = (
                f"requested {requested}, available {inventory[product]}"
            )

    if errors:
        return dict(inventory), errors

    result = dict(inventory)

    for product, requested in order.items():
        remaining = result[product] - requested

        if remaining == 0:
            result.pop(product)
        else:
            result[product] = remaining

    return result, {}

In [43]:
updated_inventory, errors = reserve_inventory(inventory, order)

assert_equal(errors, {})
assert_equal(
    updated_inventory,
    {"keyboard": 3, "mouse": 7, "monitor": 3},
)

pprint(updated_inventory)

{'keyboard': 3, 'monitor': 3, 'mouse': 7}


## Step 4 — Test a failed transaction

In [44]:
impossible_order = {
    "keyboard": 6,
    "mouse": 1,
    "speaker": 2,
}

failed_inventory, errors = reserve_inventory(inventory, impossible_order)

pprint(errors)
assert_equal(failed_inventory, inventory)

{'keyboard': 'requested 6, available 5', 'speaker': 'unknown product'}


### Best-practice note

Copy-first logic is simple and reliable for moderate-sized dictionaries.

For very large data structures, a system may instead maintain an explicit rollback log or use a database transaction. The dictionary design lesson remains the same: validate invariants and avoid partially applied updates.

# Problem 6 — Maintain a directed graph represented by adjacency dictionaries

We will represent a weighted directed graph as:

```python
{
    source: {
        destination: weight
    }
}
```

This problem develops:

- nested membership,
- `setdefault`,
- deleting edges with `pop`,
- deleting a node from every adjacency dictionary,
- avoiding mutation during iteration.

In [45]:
graph = {
    "A": {"B": 5, "C": 2},
    "B": {"C": 1},
    "C": {"A": 4},
}

## Step 1 — Add or replace an edge

`setdefault(source, {})` ensures the source node has an adjacency dictionary.

In [46]:
graph_copy = deepcopy(graph)
graph_copy.setdefault("B", {})["D"] = 7
graph_copy.setdefault("D", {})["A"] = 3

pprint(graph_copy)

{'A': {'B': 5, 'C': 2}, 'B': {'C': 1, 'D': 7}, 'C': {'A': 4}, 'D': {'A': 3}}


## Step 2 — Remove one edge safely

`pop(destination, None)` removes the edge when present and avoids an exception when absent.

In [47]:
removed_weight = graph_copy["A"].pop("C", None)
print("Removed weight:", removed_weight)

missing_weight = graph_copy["A"].pop("Z", None)
print("Missing edge result:", missing_weight)

Removed weight: 2
Missing edge result: None


## Step 3 — Remove a node completely

Removing node `C` requires two different operations:

1. remove `C` as a source,
2. remove `C` as a destination from all other nodes.

In [48]:
node_to_remove = "C"
graph_without_c = deepcopy(graph)

graph_without_c.pop(node_to_remove, None)

for neighbours in graph_without_c.values():
    neighbours.pop(node_to_remove, None)

pprint(graph_without_c)

{'A': {'B': 5}, 'B': {}}


## Complete graph utility functions

In [49]:
def add_edge(
    graph: dict[str, dict[str, float]],
    source: str,
    destination: str,
    weight: float,
) -> None:
    if weight < 0:
        raise ValueError("Weight must be non-negative.")

    graph.setdefault(source, {})[destination] = weight
    graph.setdefault(destination, {})


def remove_edge(
    graph: dict[str, dict[str, float]],
    source: str,
    destination: str,
) -> float | None:
    neighbours = graph.get(source)

    if neighbours is None:
        return None

    return neighbours.pop(destination, None)


def remove_node(
    graph: dict[str, dict[str, float]],
    node: str,
) -> dict[str, float]:
    outgoing = graph.pop(node, {})

    for neighbours in graph.values():
        neighbours.pop(node, None)

    return outgoing

In [50]:
test_graph = deepcopy(graph)

add_edge(test_graph, "B", "D", 7)
add_edge(test_graph, "D", "A", 3)

assert_equal(test_graph["B"]["D"], 7)
assert "D" in test_graph

assert_equal(remove_edge(test_graph, "A", "C"), 2)
assert_equal(remove_edge(test_graph, "A", "missing"), None)

outgoing = remove_node(test_graph, "B")
assert_equal(outgoing, {"C": 1, "D": 7})
assert all("B" not in neighbours for neighbours in test_graph.values())

pprint(test_graph)

{'A': {}, 'C': {'A': 4}, 'D': {'A': 3}}


### Representation invariant

A useful invariant is:

> Every node appears as a top-level key, even when it has no outgoing edges.

The `add_edge` function preserves this by creating an empty dictionary for the destination node.

# Problem 7 — Build an inverted index for document search

An inverted index maps each token to the documents containing it.

We will build:

```python
{
    token: {
        document_id: [positions...]
    }
}
```

This representation supports:

- document frequency,
- occurrence counts,
- phrase matching,
- result ranking.

This problem develops deeply nested `setdefault` operations and dictionaries containing lists.

In [51]:
corpus = {
    "doc-1": "red blue red green",
    "doc-2": "blue green blue",
    "doc-3": "green red yellow",
}

## Step 1 — Index one document

Use `enumerate` so that every token is paired with its position.

In [52]:
one_document_index = {}

for position, token in enumerate(corpus["doc-1"].split()):
    one_document_index.setdefault(token, []).append(position)

pprint(one_document_index)

{'blue': [1], 'green': [3], 'red': [0, 2]}


## Step 2 — Add the document-id level

Now each token maps to another dictionary.

In [53]:
index = {}

for document_id, text in corpus.items():
    for position, token in enumerate(text.split()):
        index            .setdefault(token, {})            .setdefault(document_id, [])            .append(position)

pprint(index)

{'blue': {'doc-1': [1], 'doc-2': [0, 2]},
 'green': {'doc-1': [3], 'doc-2': [1], 'doc-3': [0]},
 'red': {'doc-1': [0, 2], 'doc-3': [1]},
 'yellow': {'doc-3': [2]}}


## Step 3 — Query document frequency

Document frequency is the number of document keys stored under a token.

In [54]:
document_frequency = {
    token: len(postings)
    for token, postings in index.items()
}

pprint(document_frequency)

{'blue': 2, 'green': 3, 'red': 2, 'yellow': 1}


## Step 4 — Query total occurrences

Each postings list stores positions. Summing their lengths gives total occurrences.

In [55]:
total_occurrences = {
    token: sum(len(positions) for positions in postings.values())
    for token, postings in index.items()
}

pprint(total_occurrences)

{'blue': 3, 'green': 3, 'red': 3, 'yellow': 1}


## Step 5 — Find documents containing all query tokens

We can intersect the document-id sets.

In [56]:
def documents_containing_all(
    index: Mapping[str, Mapping[str, list[int]]],
    query_tokens: Iterable[str],
) -> set[str]:
    query_tokens = list(query_tokens)

    if not query_tokens:
        return set()

    first = query_tokens[0]

    if first not in index:
        return set()

    matches = set(index[first])

    for token in query_tokens[1:]:
        if token not in index:
            return set()

        matches.intersection_update(index[token])

    return matches

In [57]:
assert_equal(
    documents_containing_all(index, ["red", "green"]),
    {"doc-1", "doc-3"},
)
assert_equal(
    documents_containing_all(index, ["blue", "yellow"]),
    set(),
)

## Complete reusable index builder

In [58]:
def build_inverted_index(
    documents: Mapping[str, str],
) -> dict[str, dict[str, list[int]]]:
    result: dict[str, dict[str, list[int]]] = {}

    for document_id, text in documents.items():
        for position, token in enumerate(normalize_tokens(text)):
            (
                result
                .setdefault(token, {})
                .setdefault(document_id, [])
                .append(position)
            )

    return result

In [59]:
full_index = build_inverted_index(corpus)
assert_equal(full_index, index)
pprint(full_index)

{'blue': {'doc-1': [1], 'doc-2': [0, 2]},
 'green': {'doc-1': [3], 'doc-2': [1], 'doc-3': [0]},
 'red': {'doc-1': [0, 2], 'doc-3': [1]},
 'yellow': {'doc-3': [2]}}


### Best-practice note

Lists preserve every occurrence position. Sets would remove duplicates and lose ordering.

Choose the value type based on the question the data structure must answer.

# Problem 8 — Use dictionary insertion order for an undo history

Modern Python dictionaries preserve insertion order.

`popitem()` removes and returns the most recently inserted key-value pair. This gives dictionary-based LIFO behavior.

We will build a small document state with undo support.

In [60]:
document = {
    "title": "Draft",
    "status": "open",
    "priority": "normal",
}

history = {}

## Step 1 — Record the previous value before each change

A unique sequence number is used as the history key. The history value describes how to undo the operation.

In [61]:
sequence = 0

sequence += 1
history[sequence] = ("set", "title", document["title"])
document["title"] = "Final draft"

sequence += 1
history[sequence] = ("set", "priority", document["priority"])
document["priority"] = "high"

pprint(document)
pprint(history)

{'priority': 'high', 'status': 'open', 'title': 'Final draft'}
{1: ('set', 'title', 'Draft'), 2: ('set', 'priority', 'normal')}


## Step 2 — Undo the most recent change

`popitem()` returns `(history_key, history_value)`.

In [62]:
history_id, action = history.popitem()
operation, key, old_value = action

if operation == "set":
    document[key] = old_value

print("Undid history entry:", history_id)
pprint(document)

Undid history entry: 2
{'priority': 'normal', 'status': 'open', 'title': 'Final draft'}


## Step 3 — Support both setting and deleting keys

To undo a deletion, we must remember the deleted value.

To undo creation of a new key, we must remember that the key did not previously exist. A private sentinel object represents that state.

In [63]:
MISSING = object()

def apply_change(
    state: dict[str, object],
    history: dict[int, tuple[str, object]],
    sequence: int,
    key: str,
    value: object = MISSING,
) -> int:
    sequence += 1

    if value is MISSING:
        if key not in state:
            raise KeyError(f"Cannot delete missing key: {key!r}")

        old_value = state.pop(key)
        history[sequence] = ("restore", key, old_value)
    else:
        old_value = state.get(key, MISSING)
        state[key] = value
        history[sequence] = ("revert_set", key, old_value)

    return sequence

In [64]:
def undo_last(
    state: dict[str, object],
    history: dict[int, tuple[str, object]],
) -> int:
    history_id, action = history.popitem()
    operation, key, old_value = action

    if operation == "restore":
        state[key] = old_value
    elif operation == "revert_set":
        if old_value is MISSING:
            state.pop(key, None)
        else:
            state[key] = old_value
    else:
        raise ValueError(f"Unknown undo operation: {operation!r}")

    return history_id

## Step 4 — Test a complete sequence

In [65]:
state = {"title": "Draft", "status": "open"}
undo_history = {}
sequence = 0

sequence = apply_change(state, undo_history, sequence, "title", "Published")
sequence = apply_change(state, undo_history, sequence, "owner", "Ada")
sequence = apply_change(state, undo_history, sequence, "status")

pprint(state)
pprint(undo_history)

{'owner': 'Ada', 'title': 'Published'}
{1: ('revert_set', 'title', 'Draft'),
 2: ('revert_set', 'owner', <object object at 0x0000022350C713F0>),
 3: ('restore', 'status', 'open')}


In [66]:
undo_last(state, undo_history)   # restore status
undo_last(state, undo_history)   # remove newly-created owner
undo_last(state, undo_history)   # restore title

assert_equal(state, {"title": "Draft", "status": "open"})
assert_equal(undo_history, {})

pprint(state)

{'status': 'open', 'title': 'Draft'}


### Important limitation

Assigning a new value to an existing dictionary key does not move the key to the end.

Here, that does not matter because the history dictionary receives a fresh numeric key for every operation.

# Problem 9 — Maintain a bidirectional employee index

We want efficient lookup in both directions:

- employee ID → email,
- email → employee ID.

The two dictionaries must always agree.

This problem develops:

- invariants across multiple dictionaries,
- collision validation,
- safe rename operations,
- rollback thinking.

In [67]:
by_id = {
    101: "ada@example.com",
    102: "grace@example.com",
}

by_email = {
    "ada@example.com": 101,
    "grace@example.com": 102,
}

## Step 1 — State the invariant

For every pair:

```python
employee_id -> email
```

the reverse dictionary must contain:

```python
email -> employee_id
```

If either side disagrees, the index is corrupted.

In [68]:
def validate_bidirectional_index(
    by_id: Mapping[int, str],
    by_email: Mapping[str, int],
) -> bool:
    if len(by_id) != len(by_email):
        return False

    for employee_id, email in by_id.items():
        if by_email.get(email) != employee_id:
            return False

    return True

In [69]:
assert validate_bidirectional_index(by_id, by_email)

## Step 2 — Add a record only after checking both collision types

In [70]:
def add_employee(
    by_id: dict[int, str],
    by_email: dict[str, int],
    employee_id: int,
    email: str,
) -> None:
    if employee_id in by_id:
        raise KeyError(f"Employee ID already exists: {employee_id}")

    if email in by_email:
        raise KeyError(f"Email already exists: {email}")

    by_id[employee_id] = email
    by_email[email] = employee_id

In [71]:
ids = dict(by_id)
emails = dict(by_email)

add_employee(ids, emails, 103, "linus@example.com")

assert_equal(ids[103], "linus@example.com")
assert_equal(emails["linus@example.com"], 103)
assert validate_bidirectional_index(ids, emails)

## Step 3 — Rename an email safely

Validation occurs before mutation. Then the old reverse entry is removed with `pop`.

In [72]:
def change_employee_email(
    by_id: dict[int, str],
    by_email: dict[str, int],
    employee_id: int,
    new_email: str,
) -> str:
    if employee_id not in by_id:
        raise KeyError(f"Unknown employee ID: {employee_id}")

    existing_owner = by_email.get(new_email)

    if existing_owner is not None and existing_owner != employee_id:
        raise KeyError(f"Email belongs to employee {existing_owner}: {new_email}")

    old_email = by_id[employee_id]

    if old_email == new_email:
        return old_email

    by_email.pop(old_email)
    by_id[employee_id] = new_email
    by_email[new_email] = employee_id

    return old_email

In [73]:
old_email = change_employee_email(
    ids,
    emails,
    103,
    "torvalds@example.com",
)

print("Old email:", old_email)
assert "linus@example.com" not in emails
assert_equal(emails["torvalds@example.com"], 103)
assert validate_bidirectional_index(ids, emails)

Old email: linus@example.com


## Step 4 — Remove a record

`pop` both removes and returns the associated value, so it is ideal here.

In [74]:
def remove_employee(
    by_id: dict[int, str],
    by_email: dict[str, int],
    employee_id: int,
) -> str:
    email = by_id.pop(employee_id)
    removed_id = by_email.pop(email)

    assert removed_id == employee_id
    return email

In [75]:
removed_email = remove_employee(ids, emails, 102)

assert_equal(removed_email, "grace@example.com")
assert validate_bidirectional_index(ids, emails)

pprint(ids)
pprint(emails)

{101: 'ada@example.com', 103: 'torvalds@example.com'}
{'ada@example.com': 101, 'torvalds@example.com': 103}


### Best-practice note

When multiple dictionaries represent one logical data structure, expose operations through functions or a class. Allowing arbitrary external mutation makes it easy to violate the invariant.

# Problem 10 — Deeply merge nested settings with conflict policies

A shallow `dict.update` replaces an entire nested dictionary.

Sometimes we instead want recursive merging:

- dictionary + dictionary → merge recursively,
- any other conflict → later value replaces earlier value.

We will also add an optional conflict policy.

In [76]:
base_settings = {
    "database": {
        "host": "db.internal",
        "port": 5432,
        "options": {
            "ssl": True,
            "pool_size": 10,
        },
    },
    "features": {
        "search": True,
        "recommendations": False,
    },
}

override_settings = {
    "database": {
        "port": 6432,
        "options": {
            "pool_size": 20,
        },
    },
    "features": {
        "recommendations": True,
    },
}

## Step 1 — Observe shallow update behavior

In [77]:
shallow = deepcopy(base_settings)
shallow.update(override_settings)

pprint(shallow)

{'database': {'options': {'pool_size': 20}, 'port': 6432},
 'features': {'recommendations': True}}


The original `database["host"]` and `database["options"]["ssl"]` settings disappeared because the whole `database` value was replaced.

## Step 2 — Write the recursive case

When both current and incoming values are dictionaries, recurse. Otherwise, replace.

In [78]:
def deep_merge(
    base: Mapping[str, object],
    override: Mapping[str, object],
) -> dict[str, object]:
    result = deepcopy(dict(base))

    for key, incoming_value in override.items():
        current_value = result.get(key)

        if isinstance(current_value, Mapping) and isinstance(incoming_value, Mapping):
            result[key] = deep_merge(current_value, incoming_value)
        else:
            result[key] = deepcopy(incoming_value)

    return result

In [79]:
merged = deep_merge(base_settings, override_settings)
pprint(merged)

{'database': {'host': 'db.internal',
              'options': {'pool_size': 20, 'ssl': True},
              'port': 6432},
 'features': {'recommendations': True, 'search': True}}


In [80]:
assert_equal(merged["database"]["host"], "db.internal")
assert_equal(merged["database"]["port"], 6432)
assert_equal(merged["database"]["options"]["ssl"], True)
assert_equal(merged["database"]["options"]["pool_size"], 20)
assert_equal(merged["features"]["recommendations"], True)

## Step 3 — Confirm the inputs were not mutated

In [81]:
assert_equal(base_settings["database"]["port"], 5432)
assert_equal(override_settings["database"]["port"], 6432)

## Step 4 — Add a strict conflict policy

In strict mode, replacing a mapping with a non-mapping, or a non-mapping with a mapping, is rejected.

In [82]:
def deep_merge_strict(
    base: Mapping[str, object],
    override: Mapping[str, object],
    *,
    path: tuple[str, ...] = (),
) -> dict[str, object]:
    result = deepcopy(dict(base))

    for key, incoming_value in override.items():
        current_path = path + (key,)

        if key not in result:
            result[key] = deepcopy(incoming_value)
            continue

        current_value = result[key]
        current_is_mapping = isinstance(current_value, Mapping)
        incoming_is_mapping = isinstance(incoming_value, Mapping)

        if current_is_mapping and incoming_is_mapping:
            result[key] = deep_merge_strict(
                current_value,
                incoming_value,
                path=current_path,
            )
        elif current_is_mapping != incoming_is_mapping:
            dotted_path = ".".join(current_path)
            raise TypeError(
                f"Incompatible values at {dotted_path}: "
                f"{type(current_value).__name__} vs "
                f"{type(incoming_value).__name__}"
            )
        else:
            result[key] = deepcopy(incoming_value)

    return result

In [83]:
bad_override = {
    "database": "use-default-database",
}

try:
    deep_merge_strict(base_settings, bad_override)
except TypeError as exc:
    print(exc)

Incompatible values at database: dict vs str


# Problem 11 — Compute a recursive dictionary diff and apply a patch

We want to compare two nested dictionaries and describe:

- added keys,
- removed keys,
- changed values.

Then we will apply a patch to reconstruct the target dictionary.

This problem combines membership, recursion, `pop`, and nested updates.

In [84]:
before = {
    "name": "analytics-service",
    "replicas": 2,
    "resources": {
        "cpu": "500m",
        "memory": "512Mi",
    },
    "labels": {
        "team": "data",
        "tier": "backend",
    },
}

after = {
    "name": "analytics-service",
    "replicas": 4,
    "resources": {
        "cpu": "1",
        "memory": "512Mi",
    },
    "labels": {
        "team": "platform",
    },
    "autoscaling": True,
}

## Step 1 — Compare top-level key sets

Keys can be partitioned into:

- only in `before`,
- only in `after`,
- in both.

In [85]:
before_keys = set(before)
after_keys = set(after)

print("Added:", after_keys - before_keys)
print("Removed:", before_keys - after_keys)
print("Shared:", before_keys & after_keys)

Added: {'autoscaling'}
Removed: set()
Shared: {'resources', 'replicas', 'labels', 'name'}


## Step 2 — Define a patch format

We will use:

```python
{
    "added": {path_tuple: value},
    "removed": {path_tuple: old_value},
    "changed": {path_tuple: (old_value, new_value)},
}
```

Paths are tuples such as `("resources", "cpu")`.

In [86]:
def dictionary_diff(
    before: Mapping[str, object],
    after: Mapping[str, object],
    *,
    path: tuple[str, ...] = (),
) -> dict[str, dict[tuple[str, ...], object]]:
    diff = {
        "added": {},
        "removed": {},
        "changed": {},
    }

    before_keys = set(before)
    after_keys = set(after)

    for key in after_keys - before_keys:
        diff["added"][path + (key,)] = deepcopy(after[key])

    for key in before_keys - after_keys:
        diff["removed"][path + (key,)] = deepcopy(before[key])

    for key in before_keys & after_keys:
        old_value = before[key]
        new_value = after[key]
        current_path = path + (key,)

        if isinstance(old_value, Mapping) and isinstance(new_value, Mapping):
            nested = dictionary_diff(
                old_value,
                new_value,
                path=current_path,
            )

            for category in diff:
                diff[category].update(nested[category])
        elif old_value != new_value:
            diff["changed"][current_path] = (
                deepcopy(old_value),
                deepcopy(new_value),
            )

    return diff

In [87]:
diff = dictionary_diff(before, after)
pprint(diff)

{'added': {('autoscaling',): True},
 'changed': {('labels', 'team'): ('data', 'platform'),
             ('replicas',): (2, 4),
             ('resources', 'cpu'): ('500m', '1')},
 'removed': {('labels', 'tier'): 'backend'}}


## Step 3 — Write helpers for nested paths

To apply the patch, we need to navigate to the parent dictionary of a path.

In [88]:
def get_parent_dictionary(
    root: dict[str, object],
    path: tuple[str, ...],
) -> tuple[dict[str, object], str]:
    if not path:
        raise ValueError("Path cannot be empty.")

    current = root

    for key in path[:-1]:
        child = current[key]

        if not isinstance(child, dict):
            raise TypeError(f"Path component is not a dictionary: {key!r}")

        current = child

    return current, path[-1]

## Step 4 — Apply removals, changes, and additions

In [89]:
def apply_dictionary_diff(
    source: Mapping[str, object],
    diff: Mapping[str, Mapping[tuple[str, ...], object]],
) -> dict[str, object]:
    result = deepcopy(dict(source))

    for path in sorted(
        diff["removed"],
        key=len,
        reverse=True,
    ):
        parent, key = get_parent_dictionary(result, path)
        parent.pop(key)

    for path, change in diff["changed"].items():
        _, new_value = change
        parent, key = get_parent_dictionary(result, path)
        parent[key] = deepcopy(new_value)

    for path, value in sorted(diff["added"].items(), key=lambda item: len(item[0])):
        parent, key = get_parent_dictionary(result, path)
        parent[key] = deepcopy(value)

    return result

In [90]:
reconstructed = apply_dictionary_diff(before, diff)

assert_equal(reconstructed, after)
pprint(reconstructed)

{'autoscaling': True,
 'labels': {'team': 'platform'},
 'name': 'analytics-service',
 'replicas': 4,
 'resources': {'cpu': '1', 'memory': '512Mi'}}


### Why removals are processed deepest-first

Suppose both a parent path and one of its child paths were removed. Removing the parent first would make the child path inaccessible.

Deepest-first processing avoids that issue.

# Problem 12 — Aggregate event streams into user sessions

Each event contains:

- a user,
- a timestamp,
- an action.

Events for the same user belong to the same session when the gap is at most 30 minutes.

We want a dictionary:

```python
{
    user: [
        {
            "start": ...,
            "end": ...,
            "actions": [...]
        },
        ...
    ]
}
```

This final problem combines grouping, nested mutation, safe access, and careful ordering.

In [91]:
events = [
    {"user": "ada", "time": 0, "action": "login"},
    {"user": "grace", "time": 3, "action": "login"},
    {"user": "ada", "time": 10, "action": "view-dashboard"},
    {"user": "ada", "time": 50, "action": "export-report"},
    {"user": "grace", "time": 20, "action": "run-query"},
    {"user": "grace", "time": 65, "action": "logout"},
    {"user": "ada", "time": 70, "action": "logout"},
]

## Step 1 — Sort events

Sessionization depends on chronological order. We should not assume the input is already sorted.

In [92]:
ordered_events = sorted(events, key=lambda event: event["time"])
pprint(ordered_events)

[{'action': 'login', 'time': 0, 'user': 'ada'},
 {'action': 'login', 'time': 3, 'user': 'grace'},
 {'action': 'view-dashboard', 'time': 10, 'user': 'ada'},
 {'action': 'run-query', 'time': 20, 'user': 'grace'},
 {'action': 'export-report', 'time': 50, 'user': 'ada'},
 {'action': 'logout', 'time': 65, 'user': 'grace'},
 {'action': 'logout', 'time': 70, 'user': 'ada'}]


## Step 2 — Store sessions by user

For every event:

1. get or create the user's session list,
2. inspect the latest session,
3. either extend it or begin a new session.

In [93]:
sessions = {}
maximum_gap = 30

for event in ordered_events:
    user_sessions = sessions.setdefault(event["user"], [])

    if not user_sessions:
        user_sessions.append({
            "start": event["time"],
            "end": event["time"],
            "actions": [event["action"]],
        })
        continue

    latest_session = user_sessions[-1]
    gap = event["time"] - latest_session["end"]

    if gap <= maximum_gap:
        latest_session["end"] = event["time"]
        latest_session["actions"].append(event["action"])
    else:
        user_sessions.append({
            "start": event["time"],
            "end": event["time"],
            "actions": [event["action"]],
        })

pprint(sessions)

{'ada': [{'actions': ['login', 'view-dashboard'], 'end': 10, 'start': 0},
         {'actions': ['export-report', 'logout'], 'end': 70, 'start': 50}],
 'grace': [{'actions': ['login', 'run-query'], 'end': 20, 'start': 3},
           {'actions': ['logout'], 'end': 65, 'start': 65}]}


## Step 3 — Complete reusable solution

In [94]:
def build_sessions(
    events: Iterable[Mapping[str, object]],
    *,
    maximum_gap: int = 30,
) -> dict[str, list[dict[str, object]]]:
    if maximum_gap < 0:
        raise ValueError("maximum_gap must be non-negative.")

    result: dict[str, list[dict[str, object]]] = {}

    ordered = sorted(events, key=lambda event: event["time"])

    for event in ordered:
        user = str(event["user"])
        timestamp = int(event["time"])
        action = str(event["action"])

        user_sessions = result.setdefault(user, [])

        if user_sessions:
            latest = user_sessions[-1]
            gap = timestamp - latest["end"]

            if gap <= maximum_gap:
                latest["end"] = timestamp
                latest["actions"].append(action)
                continue

        user_sessions.append({
            "start": timestamp,
            "end": timestamp,
            "actions": [action],
        })

    return result

In [95]:
session_result = build_sessions(events, maximum_gap=30)

assert_equal(len(session_result["ada"]), 2)
assert_equal(len(session_result["grace"]), 2)
assert_equal(
    session_result["ada"][0]["actions"],
    ["login", "view-dashboard"],
)
assert_equal(
    session_result["ada"][1]["actions"],
    ["export-report", "logout"],
)

pprint(session_result)

{'ada': [{'actions': ['login', 'view-dashboard'], 'end': 10, 'start': 0},
         {'actions': ['export-report', 'logout'], 'end': 70, 'start': 50}],
 'grace': [{'actions': ['login', 'run-query'], 'end': 20, 'start': 3},
           {'actions': ['logout'], 'end': 65, 'start': 65}]}


## Step 4 — Derive a per-user summary

Once the nested representation exists, additional reports become easy to express.

In [96]:
session_statistics = {}

for user, user_sessions in session_result.items():
    total_actions = sum(
        len(session["actions"])
        for session in user_sessions
    )
    total_duration = sum(
        session["end"] - session["start"]
        for session in user_sessions
    )

    session_statistics[user] = {
        "session_count": len(user_sessions),
        "action_count": total_actions,
        "total_duration": total_duration,
    }

pprint(session_statistics)

{'ada': {'action_count': 4, 'session_count': 2, 'total_duration': 30},
 'grace': {'action_count': 3, 'session_count': 2, 'total_duration': 17}}


# Additional worked mini-examples

The following shorter examples reinforce important dictionary behaviors.

## Mini-example A — Count transitions between consecutive states

In [97]:
states = ["idle", "running", "running", "paused", "running", "stopped"]

transition_counts = {}

for source, destination in zip(states, states[1:]):
    transition = (source, destination)
    transition_counts[transition] = transition_counts.get(transition, 0) + 1

pprint(transition_counts)

{('idle', 'running'): 1,
 ('paused', 'running'): 1,
 ('running', 'paused'): 1,
 ('running', 'running'): 1,
 ('running', 'stopped'): 1}


Tuple keys are useful when a key naturally contains multiple components.

## Mini-example B — Remove empty nested buckets

In [98]:
buckets = {
    "errors": ["E1"],
    "warnings": [],
    "info": ["I1", "I2"],
    "debug": [],
}

for key in list(buckets):
    if not buckets[key]:
        buckets.pop(key)

pprint(buckets)

{'errors': ['E1'], 'info': ['I1', 'I2']}


## Mini-example C — Drain a dictionary in LIFO order

In [99]:
tasks = {
    "task-1": "download",
    "task-2": "transform",
    "task-3": "upload",
}

drained = []

while tasks:
    drained.append(tasks.popitem())

pprint(drained)
assert_equal(
    [task_id for task_id, _ in drained],
    ["task-3", "task-2", "task-1"],
)

[('task-3', 'upload'), ('task-2', 'transform'), ('task-1', 'download')]


## Mini-example D — Clear a dictionary while preserving aliases

In [100]:
cache = {"a": 1, "b": 2}
cache_alias = cache

cache.clear()

print("cache:", cache)
print("cache_alias:", cache_alias)
assert cache_alias is cache
assert_equal(cache_alias, {})

cache: {}
cache_alias: {}


Reassigning `cache = {}` would create a new dictionary and would not empty `cache_alias`.

Use `clear()` when all references to the same dictionary should observe the reset.

# Final review

This notebook used dictionaries as:

- frequency tables,
- layered configurations,
- nested groupings,
- sparse numeric structures,
- transactional inventory state,
- graph adjacency maps,
- inverted indexes,
- ordered undo histories,
- bidirectional indexes,
- recursive configuration trees,
- structural patches,
- session aggregators.

The most important design lessons are:

1. Use `get` when missing keys have a natural default.
2. Use direct indexing when missing keys should be treated as errors.
3. Use `in` when presence matters independently of the stored value.
4. Use `setdefault` when a missing key should receive a mutable container.
5. Use `pop` when removal and retrieval belong to the same operation.
6. Use `popitem` for last-in-first-out processing of insertion-ordered items.
7. Use `clear` when aliases must observe the same dictionary becoming empty.
8. Validate invariants before mutating related dictionaries.
9. Avoid mutating dictionary size while iterating directly over it.
10. Choose list, set, tuple, or nested-dictionary values according to the questions your program must answer.

## Suggested independent extensions

Try extending the solved problems without changing their core representations:

- add stop-word filtering to the token counter,
- support environment-variable type conversion in configuration merging,
- calculate error rates from the log summary,
- normalize sparse vectors,
- support releasing a previous inventory reservation,
- compute graph in-degrees and out-degrees,
- implement phrase search with the inverted index,
- add redo support to the undo history,
- wrap the bidirectional index in a class,
- support list-merging policies in `deep_merge`,
- invert a dictionary diff,
- compute average session duration per user.

These extensions are intentionally left as practice after the complete worked solutions.